# 02 · Stage 2: column statistics

For every window the Left rows are pulled from the source and the Right rows from the warehouse
(Statement Execution `JSON_ARRAY` returns values as text), both use the same query-side key hash, and both
get the same statistics (`n_total`, `n_missing`, `n_unique`, mean/std, min/max, lengths, hash sum).
The Left producer and Right warehouse consumer run together: Right prefetch starts immediately,
and each Unit is compared as soon as its `_INDEX` pointer appears.

The pulled windows are also kept in `<run>/_local/data.duckdb` (tables `left__<pair>`,
`right__<pair>`, column `dtrack_bucket`). `PUSH_SAMPLE=False` retains a full Right window for
diagnostics; built-in Left key families still apply their deterministic source predicate.
`ERROR` and `CRITICAL` details go to `<run>/_local/logs/errors.log`, not the notebook console.

In [ ]:
# Parameters: keep CONFIG / OUT identical in notebooks 00-03
CONFIG = "c.json"  # pairs config (col_map files resolve next to it)
OUT = ".dtrack-local/runs"  # local runs root; one folder per run id
RUN_ID = None  # None: 00 starts a new run, 01-03 continue the run 00 opened (OUT/LATEST)
DISPOSITION = (
    "external_links"  # "inline" when presigned downloads are blocked (<= 25 MiB per window)
)
PUSH_SAMPLE = True  # False only when you need to land the full Right window locally
PAIRS = None  # e.g. ["orders"]; None = every active pair

In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    stream=sys.stdout,
    force=True,
)
pd.set_option("display.max_columns", 40, "display.width", 200)

from dtrack_local import pipeline

In [ ]:
RUN_ID = RUN_ID or (Path(OUT) / "LATEST").read_text().strip()
run = pipeline.open_run(
    CONFIG, run_id=RUN_ID, out=OUT, disposition=DISPOSITION, push_sample=PUSH_SAMPLE, pairs=PAIRS
)
print("run_id:", run.run_id)
print("folder:", run.folder.resolve())
print("status:", run.status())

In [ ]:
assert run.store.exists("row_check/_CLOSED"), "run notebook 01 first"
stage_two = pipeline.col_check(run)
stage_two

## Results
Differences first.

In [ ]:
results = pd.DataFrame([row for _, row in run.store.list_json("results/")])
if results.empty:
    print("no units compared")
else:
    view = [
        "pair",
        "vintage",
        "column",
        "col_type",
        "verdict",
        "n_total_left",
        "n_total_right",
        "n_missing_left",
        "n_missing_right",
        "n_unique_left",
        "n_unique_right",
        "min_left",
        "min_right",
        "max_left",
        "max_right",
        "hash_sum_match",
    ]
    display(results[view].sort_values(["verdict", "pair", "vintage", "column"]))

## Dead units
A unit is dead after three failed attempts; re-running this notebook retries nothing that finished.

In [ ]:
dead = [row for _, row in run.store.list_json("dead/")]
pd.DataFrame(dead)[["unit_id", "error"]] if dead else "no dead units"

## The pulled rows in DuckDB

In [ ]:
import duckdb

print(run.landing.tables())
database = str(run.landing.path)
with duckdb.connect(database, read_only=True) as connection:
    tables = [row[0] for row in connection.execute("SHOW TABLES").fetchall()]
    for pair in sorted({name.split("__", 1)[1] for name in tables}):
        parts = [
            f"SELECT '{side}' AS side, dtrack_bucket, count(*) AS rows FROM \"{side}__{pair}\" GROUP BY ALL"
            for side in ("left", "right")
            if f"{side}__{pair}" in tables
        ]
        print(pair)
        display(connection.execute(" UNION ALL ".join(parts) + " ORDER BY 2, 1").df())
print(f"open it yourself: duckdb -readonly {database}")